# Taxis NYC: Pronóstico, Segmentación y Decisión Operativa
**Notebook de preparación** — ETL, modelo de pronóstico por cuantiles (XGBoost) y segmentación de zonas (K-Means).

**Reglas de negocio**
1. Se descartan las zonas *Unknown* (`PULocationID` 264 y 265).
2. Segmentación: zonas con **≥ 500** viajes en enero 2024.
3. Pronóstico: zonas con **≥ 3,000** viajes en enero 2024.
4. Partición temporal estricta (sin `train_test_split`): entrenamiento 8–24 ene, test 25–31 ene, pronóstico 1 feb 2024 ("mañana").
5. Sin fuga de datos: solo lags ≥ 24 h (24, 48, 167, 168, 169) y la media de las 24 h del día anterior.

**Salidas**
- `api/modelo_pronostico.ubj` — modelo XGBoost multi-cuantil.
- `api/demanda.parquet` — features + demanda real del 25 ene al 1 feb.
- `app/data/segmentacion_zonas.csv` — segmentos, PCA y perfil horario de 24 h.
- `app/data/metricas_modelo.json` — métricas para la ficha técnica y el monitoreo.

## 0. Librerías y configuración

In [1]:
import io
import json
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)
RAIZ = Path.cwd()
DIR_API = RAIZ / "api"
DIR_APP_DATA = RAIZ / "app" / "data"
DIR_API.mkdir(exist_ok=True)
DIR_APP_DATA.mkdir(parents=True, exist_ok=True)

URL_VIAJES = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
URL_ZONAS = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

ZONAS_UNKNOWN = [264, 265]
MIN_VIAJES_SEGMENTACION = 500
MIN_VIAJES_PRONOSTICO = 3_000
INICIO_TRAIN, FIN_TRAIN = pd.Timestamp("2024-01-08"), pd.Timestamp("2024-01-24 23:00")
INICIO_TEST, FIN_TEST = pd.Timestamp("2024-01-25"), pd.Timestamp("2024-01-31 23:00")
DIA_FUTURO = pd.Timestamp("2024-02-01")
FIN_SERIE = pd.Timestamp("2024-02-01 23:00")
CUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
SEMILLA = 42
print("xgboost", xgb.__version__, "| pandas", pd.__version__)

xgboost 3.2.0 | pandas 2.3.3


## 1. Carga de datos
Los datos se leen directamente en memoria desde la URL oficial de TLC (no se guarda el parquet crudo de ~50 MB en el repositorio).

In [2]:
COLUMNAS = ["tpep_pickup_datetime", "PULocationID", "trip_distance", "fare_amount", "tip_amount"]
with urllib.request.urlopen(URL_VIAJES) as r:
    viajes_raw = pd.read_parquet(io.BytesIO(r.read()), columns=COLUMNAS)
zonas = pd.read_csv(URL_ZONAS)
print(viajes_raw.shape)
viajes_raw.head()

(2964624, 5)


,tpep_pickup_datetime,PULocationID,trip_distance,fare_amount,tip_amount
0,2024-01-01 00:57:55,186,1.72,17.7,0.00
1,2024-01-01 00:03:00,140,1.80,10.0,3.75
2,2024-01-01 00:17:06,236,4.70,23.3,3.00
3,2024-01-01 00:36:38,79,1.40,10.0,2.00
4,2024-01-01 00:46:51,211,0.80,7.9,3.20


## 2. Limpieza y reglas de negocio

In [3]:
viajes = viajes_raw[
    (viajes_raw["tpep_pickup_datetime"] >= "2024-01-01")
    & (viajes_raw["tpep_pickup_datetime"] < "2024-02-01")
    & (~viajes_raw["PULocationID"].isin(ZONAS_UNKNOWN))
].copy()
print(f"Viajes fuera de enero o en zonas Unknown descartados: {len(viajes_raw) - len(viajes):,}")

viajes_por_zona = viajes["PULocationID"].value_counts()
zonas_segmentacion = sorted(viajes_por_zona[viajes_por_zona >= MIN_VIAJES_SEGMENTACION].index)
zonas_pronostico = sorted(viajes_por_zona[viajes_por_zona >= MIN_VIAJES_PRONOSTICO].index)
assert not set(ZONAS_UNKNOWN) & set(zonas_segmentacion)
print(f"Viajes válidos en enero: {len(viajes):,}")
print(f"Zonas para segmentación (>= {MIN_VIAJES_SEGMENTACION}): {len(zonas_segmentacion)}")
print(f"Zonas para pronóstico (>= {MIN_VIAJES_PRONOSTICO:,}): {len(zonas_pronostico)}")

Viajes fuera de enero o en zonas Unknown descartados: 12,036
Viajes válidos en enero: 2,952,588
Zonas para segmentación (>= 500): 100
Zonas para pronóstico (>= 3,000): 58


## 3. Serie temporal horaria por zona
Se agrupa por `PULocationID` y hora (`dt.floor('h')`) y se completan las combinaciones faltantes con 0 mediante `pd.MultiIndex.from_product` hasta el **1 feb 2024 23:00**. Las 24 horas del 1 de febrero no tienen dato real: tras construir los lags se marcan como `NaN`.

In [4]:
viajes["hora_viaje"] = viajes["tpep_pickup_datetime"].dt.floor("h")
conteo = (
    viajes[viajes["PULocationID"].isin(zonas_pronostico)]
    .groupby(["PULocationID", "hora_viaje"]).size().rename("viajes")
)
horas = pd.date_range("2024-01-01 00:00", FIN_SERIE, freq="h")
indice = pd.MultiIndex.from_product([zonas_pronostico, horas], names=["PULocationID", "hora_viaje"])
serie = conteo.reindex(indice, fill_value=0).reset_index()
serie["viajes"] = serie["viajes"].astype(float)
print(serie.shape, "=", len(zonas_pronostico), "zonas x", len(horas), "horas")
serie.head()

(44544, 3) = 58 zonas x 768 horas


,PULocationID,hora_viaje,viajes
0,4,2024-01-01 00:00:00,25.0
1,4,2024-01-01 01:00:00,29.0
2,4,2024-01-01 02:00:00,34.0
3,4,2024-01-01 03:00:00,31.0
4,4,2024-01-01 04:00:00,32.0


## 4. Ingeniería de variables (sin fuga de datos)
Solo se usan valores con al menos 24 h de antigüedad: lags 24, 48, 167, 168, 169 y la media de las 24 horas del día calendario anterior.

In [5]:
serie = serie.sort_values(["PULocationID", "hora_viaje"]).reset_index(drop=True)
g = serie.groupby("PULocationID")["viajes"]
LAGS = [24, 48, 167, 168, 169]
for k in LAGS:
    serie[f"lag_{k}"] = g.shift(k)

serie["fecha"] = serie["hora_viaje"].dt.normalize()
media_diaria = serie.groupby(["PULocationID", "fecha"])["viajes"].mean().rename("media_dia_anterior").reset_index()
media_diaria["fecha"] = media_diaria["fecha"] + pd.Timedelta(days=1)
serie = serie.merge(media_diaria, on=["PULocationID", "fecha"], how="left")

serie["hora"] = serie["hora_viaje"].dt.hour
serie["dia_semana"] = serie["hora_viaje"].dt.dayofweek
serie["es_fin_semana"] = (serie["dia_semana"] >= 5).astype(int)
serie["zona"] = pd.Categorical(serie["PULocationID"], categories=zonas_pronostico)

# El 1 de febrero es el "mañana": no existe demanda real.
serie.loc[serie["fecha"] == DIA_FUTURO, "viajes"] = np.nan

# Verificación de no fuga: el lag más corto apunta 24 h atrás
muestra = serie[serie["PULocationID"] == zonas_pronostico[0]].set_index("hora_viaje")
t = pd.Timestamp("2024-01-20 15:00")
assert muestra.loc[t, "lag_24"] == muestra.loc[t - pd.Timedelta(hours=24), "viajes"]
assert serie.loc[serie["fecha"] == DIA_FUTURO, "lag_24"].notna().all()

FEATURES = ["zona", "hora", "dia_semana", "es_fin_semana",
            "lag_24", "lag_48", "lag_167", "lag_168", "lag_169", "media_dia_anterior"]
serie[["PULocationID", "hora_viaje", "viajes"] + FEATURES[1:]].tail()

,PULocationID,hora_viaje,viajes,hora,dia_semana,es_fin_semana,lag_24,lag_48,lag_167,lag_168,lag_169,media_dia_anterior
44539,263,2024-02-01 19:00:00,NaN,19,3,0,106.0,115.0,110.0,84.0,140.0,80.833333
44540,263,2024-02-01 20:00:00,NaN,20,3,0,104.0,99.0,104.0,110.0,84.0,80.833333
44541,263,2024-02-01 21:00:00,NaN,21,3,0,92.0,96.0,85.0,104.0,110.0,80.833333
44542,263,2024-02-01 22:00:00,NaN,22,3,0,80.0,63.0,66.0,85.0,104.0,80.833333
44543,263,2024-02-01 23:00:00,NaN,23,3,0,53.0,42.0,36.0,66.0,85.0,80.833333


## 5. Partición temporal estricta

In [6]:
train = serie[(serie["hora_viaje"] >= INICIO_TRAIN) & (serie["hora_viaje"] <= FIN_TRAIN)]
test = serie[(serie["hora_viaje"] >= INICIO_TEST) & (serie["hora_viaje"] <= FIN_TEST)]
futuro = serie[serie["fecha"] == DIA_FUTURO]
assert train["hora_viaje"].max() < test["hora_viaje"].min()
assert test["hora_viaje"].max() < futuro["hora_viaje"].min()
assert futuro["viajes"].isna().all()
for nombre, d in [("Entrenamiento", train), ("Test", test), ("Futuro", futuro)]:
    print(f"{nombre:14s} {d['hora_viaje'].min()} -> {d['hora_viaje'].max()}  filas={len(d):,}")

Entrenamiento  2024-01-08 00:00:00 -> 2024-01-24 23:00:00  filas=23,664
Test           2024-01-25 00:00:00 -> 2024-01-31 23:00:00  filas=9,744
Futuro         2024-02-01 00:00:00 -> 2024-02-01 23:00:00  filas=1,392


## 6. Modelo XGBoost multi-cuantil
`objective='reg:quantileerror'` con 9 cuantiles en un único modelo.

In [7]:
modelo = xgb.XGBRegressor(
    objective="reg:quantileerror",
    quantile_alpha=np.array(CUANTILES),
    n_estimators=250,
    max_depth=5,
    learning_rate=0.1,
    tree_method="hist",
    enable_categorical=True,
    random_state=SEMILLA,
)
modelo.fit(train[FEATURES], train["viajes"])

def predecir(df):
    p = modelo.predict(df[FEATURES])
    return np.clip(np.sort(p, axis=1), 0, None)  # evita cuantiles cruzados

pred_test = predecir(test)
pred_train = predecir(train)
pred_test.shape

(9744, 9)

## 7. Evaluación en test (25–31 ene)

In [8]:
def pinball(y, q_pred, q):
    e = y - q_pred
    return float(np.mean(np.maximum(q * e, (q - 1) * e)))

def resumen(y, P, base):
    p50 = P[:, CUANTILES.index(0.5)]
    return {
        "mae_p50": float(np.mean(np.abs(y - p50))),
        "rmse_p50": float(np.sqrt(np.mean((y - p50) ** 2))),
        "wape_p50": float(np.sum(np.abs(y - p50)) / np.sum(y)),
        "mae_baseline_lag168": float(np.mean(np.abs(y - base))),
        "wape_baseline_lag168": float(np.sum(np.abs(y - base)) / np.sum(y)),
        "cobertura_p10_p90": float(np.mean((y >= P[:, 0]) & (y <= P[:, -1]))),
        "pinball_promedio": float(np.mean([pinball(y, P[:, i], q) for i, q in enumerate(CUANTILES)])),
    }

y_test = test["viajes"].to_numpy()
met_test = resumen(y_test, pred_test, test["lag_168"].to_numpy())
met_train = resumen(train["viajes"].to_numpy(), pred_train, train["lag_168"].fillna(0).to_numpy())
pinball_q = {str(q): pinball(y_test, pred_test[:, i], q) for i, q in enumerate(CUANTILES)}
cobertura_q = {str(q): float(np.mean(y_test <= pred_test[:, i])) for i, q in enumerate(CUANTILES)}

pd.DataFrame({"train": met_train, "test": met_test}).round(3)

,train,test
mae_p50,10.511,11.680
rmse_p50,20.807,21.646
wape_p50,0.155,0.165
mae_baseline_lag168,18.062,11.882
wape_baseline_lag168,0.266,0.168
cobertura_p10_p90,0.819,0.819
pinball_promedio,4.121,4.574


In [9]:
pd.DataFrame({"pinball_loss": pinball_q, "cobertura_empirica": cobertura_q}).round(3)

,pinball_loss,cobertura_empirica
0.1,2.357,0.111
0.2,3.759,0.210
0.3,4.805,0.317
0.4,5.519,0.418
0.5,5.840,0.515
0.6,5.792,0.616
0.7,5.411,0.713
0.8,4.612,0.812
0.9,3.074,0.906


In [10]:
# Error diario en la semana de test (insumo de la página de monitoreo)
ev = test[["PULocationID", "fecha", "viajes"]].copy()
ev["p50"] = pred_test[:, CUANTILES.index(0.5)]
ev["abs_err"] = (ev["viajes"] - ev["p50"]).abs()
diario = ev.groupby("fecha").agg(real=("viajes", "sum"), p50=("p50", "sum"), abs_err=("abs_err", "sum"))
diario["wape"] = diario["abs_err"] / diario["real"]
diario["mae"] = ev.groupby("fecha")["abs_err"].mean()
diario.round(3)

,real,p50,abs_err,wape,mae
fecha,,,,,
2024-01-25,107641.0,107402.171875,14222.049,0.132,10.217
2024-01-26,103023.0,95044.492188,15282.420,0.148,10.979
2024-01-27,108142.0,106027.304688,19606.315,0.181,14.085
2024-01-28,90342.0,90308.492188,19237.115,0.213,13.820
2024-01-29,82177.0,78779.742188,18473.188,0.225,13.271
2024-01-30,98993.0,98148.070312,13638.668,0.138,9.798
2024-01-31,99719.0,105675.031250,13346.279,0.134,9.588


In [11]:
importancia = pd.Series(modelo.get_booster().get_score(importance_type="gain")).sort_values(ascending=False)
importancia = (importancia / importancia.sum()).round(4)
importancia

lag_168               0.5523
lag_24                0.1576
lag_169               0.0966
lag_167               0.0488
zona                  0.0396
dia_semana            0.0360
hora                  0.0342
lag_48                0.0185
media_dia_anterior    0.0163
dtype: float64

## 8. Exportación del modelo y del dataset para la API

In [12]:
modelo.save_model(DIR_API / "modelo_pronostico.ubj")

nombres = zonas.set_index("LocationID")
demanda = serie[(serie["hora_viaje"] >= INICIO_TEST) & (serie["hora_viaje"] <= FIN_SERIE)].copy()
demanda["zona_nombre"] = demanda["PULocationID"].map(nombres["Zone"])
demanda["distrito"] = demanda["PULocationID"].map(nombres["Borough"])
demanda = demanda[["PULocationID", "zona_nombre", "distrito", "hora_viaje", "viajes"] + FEATURES[1:]]
demanda.to_parquet(DIR_API / "demanda.parquet", index=False)
print(demanda.shape, demanda["hora_viaje"].min(), "->", demanda["hora_viaje"].max())

# Verificación: el modelo recargado produce las mismas predicciones
m2 = xgb.XGBRegressor()
m2.load_model(DIR_API / "modelo_pronostico.ubj")
d2 = pd.read_parquet(DIR_API / "demanda.parquet")
d2["zona"] = pd.Categorical(d2["PULocationID"], categories=sorted(d2["PULocationID"].unique()))
p_a = np.sort(m2.predict(d2[FEATURES].iloc[:24]), axis=1)
p_b = np.sort(modelo.predict(demanda.assign(zona=pd.Categorical(demanda["PULocationID"], categories=zonas_pronostico))[FEATURES].iloc[:24]), axis=1)
assert np.allclose(p_a, p_b)
print("Modelo recargado OK")

(11136, 14) 2024-01-25 00:00:00 -> 2024-02-01 23:00:00
Modelo recargado OK


## 9. Segmentación de zonas (K-Means)
Variables por zona (≥ 500 viajes): viajes/día (log), distancia mediana, tarifa mediana, propina mediana, % fin de semana, % mañana (6–11 h), % tarde (12–17 h) y % noche (20–4 h).

In [13]:
seg = viajes[viajes["PULocationID"].isin(zonas_segmentacion)].copy()
seg["hora"] = seg["tpep_pickup_datetime"].dt.hour
seg["fin_semana"] = seg["tpep_pickup_datetime"].dt.dayofweek >= 5
seg["manana"] = seg["hora"].between(6, 11)
seg["tarde"] = seg["hora"].between(12, 17)
seg["noche"] = (seg["hora"] >= 20) | (seg["hora"] <= 4)
validos = seg[(seg["trip_distance"] > 0) & (seg["trip_distance"] < 100)
              & (seg["fare_amount"] > 0) & (seg["fare_amount"] < 500) & (seg["tip_amount"] >= 0)]

perfil = pd.DataFrame({
    "viajes_mes": seg.groupby("PULocationID").size(),
    "distancia_mediana": validos.groupby("PULocationID")["trip_distance"].median(),
    "tarifa_mediana": validos.groupby("PULocationID")["fare_amount"].median(),
    "propina_mediana": validos.groupby("PULocationID")["tip_amount"].median(),
    "pct_fin_semana": seg.groupby("PULocationID")["fin_semana"].mean(),
    "pct_manana": seg.groupby("PULocationID")["manana"].mean(),
    "pct_tarde": seg.groupby("PULocationID")["tarde"].mean(),
    "pct_noche": seg.groupby("PULocationID")["noche"].mean(),
})
perfil["viajes_dia"] = perfil["viajes_mes"] / 31
perfil["log_viajes_dia"] = np.log1p(perfil["viajes_dia"])
VARS_SEG = ["log_viajes_dia", "distancia_mediana", "tarifa_mediana", "propina_mediana",
            "pct_fin_semana", "pct_manana", "pct_tarde", "pct_noche"]
X_seg = StandardScaler().fit_transform(perfil[VARS_SEG])
perfil[VARS_SEG].describe().round(2)

,log_viajes_dia,distancia_mediana,tarifa_mediana,propina_mediana,pct_fin_semana,pct_manana,pct_tarde,pct_noche
count,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
mean,5.45,3.43,20.11,1.97,0.26,0.27,0.34,0.27
std,1.96,2.78,11.99,2.06,0.09,0.12,0.09,0.15
min,2.85,1.23,10.00,0.00,0.10,0.06,0.10,0.05
25%,3.44,1.61,12.10,0.00,0.21,0.18,0.29,0.17
50%,5.41,2.44,17.00,2.44,0.25,0.27,0.35,0.24
75%,7.36,4.08,24.43,2.80,0.30,0.33,0.39,0.33
max,8.45,17.45,70.00,10.05,0.55,0.64,0.77,0.77


In [14]:
siluetas = {}
for k in range(2, 8):
    etiquetas = KMeans(n_clusters=k, n_init=20, random_state=SEMILLA).fit_predict(X_seg)
    siluetas[k] = float(silhouette_score(X_seg, etiquetas))
pd.Series(siluetas, name="silueta").round(3)

2    0.305
3    0.349
4    0.392
5    0.355
6    0.292
7    0.304
Name: silueta, dtype: float64

**Elección de k = 4.** Se evalúa el coeficiente de silueta de k=2 a 7 sobre las variables estandarizadas. El máximo se alcanza en **k = 4**, que además produce segmentos accionables para la operación: aeropuertos, núcleo de Manhattan, zonas de vida nocturna y barrios periféricos.

In [15]:
k_mejor = max(siluetas, key=siluetas.get)
print("k con mayor silueta:", k_mejor)
assert k_mejor == 4

k con mayor silueta: 4


In [16]:
K = 4
kmeans = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA)
perfil["cluster"] = kmeans.fit_predict(X_seg)
pca = PCA(n_components=2, random_state=SEMILLA)
comp = pca.fit_transform(X_seg)
perfil["pc1"], perfil["pc2"] = comp[:, 0], comp[:, 1]
print("Varianza explicada PCA:", pca.explained_variance_ratio_.round(3), "total", pca.explained_variance_ratio_.sum().round(3))

# Nombrado automático según el perfil promedio de cada cluster
centro = perfil.groupby("cluster")[VARS_SEG].mean()
nombres_cluster, restantes = {}, list(centro.index)
for nombre, var in [("Aeropuertos", "distancia_mediana"), ("Vida Nocturna", "pct_noche"),
                    ("Manhattan Central", "log_viajes_dia")]:
    c = centro.loc[restantes, var].idxmax()
    nombres_cluster[c] = nombre
    restantes.remove(c)
nombres_cluster[restantes[0]] = "Barrios Periféricos"
perfil["segmento"] = perfil["cluster"].map(nombres_cluster)
perfil.groupby("segmento")[VARS_SEG + ["viajes_mes"]].agg(["mean"]).round(2).T

Varianza explicada PCA: [0.352 0.295] total 0.647


,segmento,Aeropuertos,Barrios Periféricos,Manhattan Central,Vida Nocturna
log_viajes_dia,mean,5.75,3.52,6.97,5.46
distancia_mediana,mean,11.96,4.42,1.91,2.64
tarifa_mediana,mean,58.18,24.57,13.26,16.73
propina_mediana,mean,8.61,0.21,2.70,1.81
pct_fin_semana,mean,0.27,0.23,0.25,0.44
pct_manana,mean,0.22,0.37,0.24,0.12
pct_tarde,mean,0.35,0.31,0.40,0.20
pct_noche,mean,0.30,0.21,0.23,0.58
viajes_mes,mean,49712.80,1264.95,51594.85,21706.08


In [17]:
# Perfil horario normalizado (proporción de viajes por hora del día, suma = 1)
perfil_h = seg.groupby(["PULocationID", "hora"]).size().unstack(fill_value=0).reindex(columns=range(24), fill_value=0)
perfil_h = perfil_h.div(perfil_h.sum(axis=1), axis=0)
perfil_h.columns = [f"h{h:02d}" for h in range(24)]

salida_seg = perfil.join(perfil_h)
salida_seg.insert(0, "zona_nombre", salida_seg.index.map(nombres["Zone"]))
salida_seg.insert(1, "distrito", salida_seg.index.map(nombres["Borough"]))
salida_seg = salida_seg.reset_index().rename(columns={"index": "PULocationID"})
salida_seg = salida_seg.round(5)
salida_seg.to_csv(DIR_APP_DATA / "segmentacion_zonas.csv", index=False)
print(salida_seg.shape)
salida_seg[["PULocationID", "zona_nombre", "distrito", "segmento", "viajes_dia", "distancia_mediana", "pct_noche"]].sort_values("segmento").head(15)

(100, 41)


,PULocationID,zona_nombre,distrito,segmento,viajes_dia,distancia_mediana,pct_noche
49,138,LaGuardia Airport,Queens,Aeropuertos,2888.06452,9.490,0.27583
35,93,Flushing Meadows-Corona Park,Queens,Aeropuertos,16.45161,9.300,0.35882
47,132,JFK Airport,Queens,Aeropuertos,4685.16129,17.450,0.33693
24,70,East Elmhurst,Queens,Aeropuertos,396.29032,9.350,0.26447
2,10,Baisley Park,Queens,Aeropuertos,32.22581,14.185,0.24324
55,145,Long Island City/Hunters Point,Queens,Barrios Periféricos,89.16129,3.200,0.20586
80,226,Sunnyside,Queens,Barrios Periféricos,83.80645,3.575,0.22941
33,89,Flatbush/Ditmas Park,Brooklyn,Barrios Periféricos,19.19355,4.880,0.10588
79,225,Stuyvesant Heights,Brooklyn,Barrios Periféricos,18.80645,4.700,0.16810
59,152,Manhattanville,Manhattan,Barrios Periféricos,63.22581,2.230,0.15510


In [18]:
salida_seg.groupby(["segmento", "distrito"]).size().unstack(fill_value=0)

distrito,Bronx,Brooklyn,Manhattan,Queens
segmento,,,,
Aeropuertos,0,0,0,5
Barrios Periféricos,1,17,5,14
Manhattan Central,0,0,46,0
Vida Nocturna,0,4,8,0


## 10. Métricas para la app

In [19]:
metricas = {
    "modelo": {
        "algoritmo": "XGBoost XGBRegressor",
        "objective": "reg:quantileerror",
        "cuantiles": CUANTILES,
        "n_estimators": 250, "max_depth": 5, "learning_rate": 0.1,
        "features": FEATURES,
        "importancia_gain": importancia.to_dict(),
    },
    "datos": {
        "viajes_validos_enero": int(len(viajes)),
        "zonas_pronostico": len(zonas_pronostico),
        "zonas_segmentacion": len(zonas_segmentacion),
        "train": [str(INICIO_TRAIN.date()), str(FIN_TRAIN.date())],
        "test": [str(INICIO_TEST.date()), str(FIN_TEST.date())],
        "futuro": str(DIA_FUTURO.date()),
        "filas_train": int(len(train)), "filas_test": int(len(test)),
    },
    "test": met_test,
    "train": met_train,
    "pinball_por_cuantil": pinball_q,
    "cobertura_por_cuantil": cobertura_q,
    "error_diario_test": {str(f.date()): {k: float(v) for k, v in fila.items()} for f, fila in diario.iterrows()},
    "segmentacion": {
        "k": K,
        "silueta_por_k": {str(k): v for k, v in siluetas.items()},
        "varianza_pca": [float(v) for v in pca.explained_variance_ratio_],
        "zonas_por_segmento": salida_seg["segmento"].value_counts().to_dict(),
    },
}
(DIR_APP_DATA / "metricas_modelo.json").write_text(json.dumps(metricas, indent=2, ensure_ascii=False))
print(json.dumps(metricas["test"], indent=2))

{
  "mae_p50": 11.679601160128788,
  "rmse_p50": 21.64642290989829,
  "wape_p50": 0.1649274367958456,
  "mae_baseline_lag168": 11.881773399014778,
  "wape_baseline_lag168": 0.16778230732554922,
  "cobertura_p10_p90": 0.8186576354679803,
  "pinball_promedio": 4.57425819542195
}


## 11. Validación final de archivos (< 25 MB)

In [20]:
LIMITE_MB = 25
for p in sorted(RAIZ.rglob("*")):
    if p.is_file() and ".git" not in p.parts and ".ipynb_checkpoints" not in p.parts:
        mb = p.stat().st_size / 1e6
        assert mb < LIMITE_MB, f"{p} supera {LIMITE_MB} MB"
        print(f"{mb:8.3f} MB  {p.relative_to(RAIZ)}")

   0.101 MB  api/demanda.parquet
   0.003 MB  api/main.py
   7.677 MB  api/modelo_pronostico.ubj
   0.000 MB  api/requirements.txt
   0.000 MB  app/.streamlit/config.toml
   0.000 MB  app/.streamlit/secrets.toml.example
   0.004 MB  app/data/metricas_modelo.json
   0.033 MB  app/data/segmentacion_zonas.csv
   0.000 MB  app/requirements.txt
   0.001 MB  app/streamlit_app.py
   0.004 MB  app/utils.py
   0.024 MB  preparacion.ipynb
